# 12. Финальный LightGBM (Kaggle)

Финальная версия главной модели и оценка вместе со всеми предыдущими:

| модель | что нового | точки |
|---|---|---|
| `lgbm_final` | поправка смещения по горизонтам + для теста переобучение на train + val | все точки val и test |
| `lgbm_bc` | только поправка смещения (без переобучения) — чтобы разделить два эффекта | общий подсет test |

**Input:** датасет `vkr-transit-data`, вывод `lgbm-part-a` и `lgbm-part-b` (Your Work → Notebooks).
Accelerator — None, Internet — On. Время ~3–3,5 ч. Результат — `results_csv.zip` в Output.

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Прогнозы прошлых запусков и бейзлайны

In [ ]:
import pandas as pd
for p in sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True)):
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
logs = glob.glob("/kaggle/input/**/results/runs.csv", recursive=True)
if logs:
    pd.concat([pd.read_csv(p) for p in logs]).drop_duplicates().to_csv(f"{R}/runs.csv", index=False)
for p in glob.glob("/kaggle/input/**/results/fi_*.csv", recursive=True):
    shutil.copy(p, R)
!cd {REPO} && python src/baselines.py
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/preds/*.npz")))

## 2. Финальная модель

In [ ]:
t = time.time()
if not os.path.exists(f"{R}/preds/lgbm_final__test.npz"):
    !cd {REPO} && python src/train_lgbm.py --name lgbm_final --bias-correct --refit-for-test
print(f"lgbm_final: {(time.time() - t) / 60:.0f} мин")
t = time.time()
if not os.path.exists(f"{R}/preds/lgbm_bc__test.npz"):
    !cd {REPO} && python src/train_lgbm.py --name lgbm_bc --bias-correct --splits test --pred-stride 7
print(f"lgbm_bc: {(time.time() - t) / 60:.0f} мин")

## 3. Оценка

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

## 4. Сравнение финальной модели с первой версией

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 20)
H_ = ["горизонт h1", "горизонт h2-3", "горизонт h4-6", "горизонт h7-12", "горизонт h13-24"]
for split, subset in [("val", "все точки"), ("test", "все точки"), ("test", "общий подсет")]:
    m = pd.read_csv(f"{R}/metrics_{split}.csv")
    C = m[(m.subset == subset) & m.model.isin(["norm_level", "lgbm", "lgbm_bc", "lgbm_final"])]
    t = C[C.slice == "всё"].set_index("model")[["wape", "bias"]].join(
        C[C.slice.isin(H_)].pivot_table(index="model", columns="slice", values="wape")[H_])
    print(f"\n{split} / {subset}"); display(t.round(4).sort_values("wape"))

## 5. Архив для репозитория

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))